In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
import json

In [18]:
orig_data = pd.read_csv('data/original_data/newsroom-human-eval.csv')
print(orig_data.shape)
# unique ids
print("Unique ids: ", orig_data['ArticleID'].nunique())
orig_data.head()

(1260, 9)
Unique ids:  60


,ArticleID,System,ArticleText,SystemSummary,ArticleTitle,CoherenceRating,FluencyRating,InformativenessRating,RelevanceRating
0,2140,fragments,A worker sets up a polling station the morning...,John Avlon voter turnout in the is a sign of a...,Why has GOP turnout taken a dive?,2,2,2,2
1,2140,fragments,A worker sets up a polling station the morning...,John Avlon voter turnout in the is a sign of a...,Why has GOP turnout taken a dive?,4,5,4,4
2,2140,fragments,A worker sets up a polling station the morning...,John Avlon voter turnout in the is a sign of a...,Why has GOP turnout taken a dive?,2,3,2,3
3,2140,textrank,A worker sets up a polling station the morning...,"In New Hampshire , the same dynamic applied --...",Why has GOP turnout taken a dive?,4,5,4,5
4,2140,textrank,A worker sets up a polling station the morning...,"In New Hampshire , the same dynamic applied --...",Why has GOP turnout taken a dive?,3,2,3,2


In [13]:
print(orig_data['ArticleText'][1])
print()
print(orig_data['SystemSummary'][1])

A worker sets up a polling station the morning of the GOP primary in Florida. Fewer voters than expected turned out.</p><p>Editor&#x27;s note: John Avlon is a CNN contributor and senior political columnist for Newsweek and The Daily Beast. He is co-editor of the book &quot;Deadline Artists: America&#x27;s Greatest Newspaper Columns.&quot;</p><p>(CNN) -- Beneath Rick Santorum&#x27;s stunning three-state sweep on Tuesday stands another stubborn sign of dissatisfaction with the status quo: Republican turnout is down.</p><p>I&#x27;m talking embarrassingly, disturbingly, hey-don&#x27;t-you-know-it&#x27;s-an-election-year bad. It is a sign of a serious enthusiasm gap among the rank and file, and a particularly bad omen for Mitt Romney and the GOP in the general election.</p><p>Here&#x27;s the tale of the tape, state by state, beginning with Tuesday night: Minnesota had just more than 47,000 people turn out for its caucuses this year -- four years ago it was nearly 63,000 -- and Romney came i

In [19]:
data = json.loads(open('data/newsroom.json').read())
for idx, annot in enumerate(data['annotations'], 1):
    print(f"#{idx}")
    print(f" - Metrics: {annot['metric']}")
    print(f" - Prompt: {annot['prompt']}")
    print()
    
    with open(f"prompts/{annot['metric']}_prompt.txt", "w", encoding="utf-8") as f:
        f.write(annot['prompt'])

#1
 - Metrics: Informativeness
 - Prompt: On a scale of 1 (low) to 5 (high), how well does the summary capture the key points of the article?

{{ instance }}

#2
 - Metrics: Relevance
 - Prompt: On a scale of 1 (low) to 5 (high), are the details provided by the summary consistent with details in the article?

{{ instance }}

#3
 - Metrics: Fluency
 - Prompt: On a scale of 1 (low) to 5 (high), are the individual sentences of the summary well-written and grammatical?

{{ instance }}

#4
 - Metrics: Coherence
 - Prompt: On a scale of 1 (low) to 5 (high), do phrases and sentences of the summary fit together and make sense collectively?

{{ instance }}



In [9]:
data['instances'][0]

{'id': 1,
 'instance': '### Generated Summary\n\ncollection of all usatoday.com coverage of war of chenda , including articles , videos , photos , and quotes . and videos .\n\n### Source Article\n\n\'16 & Pregnant\' Couple Arrested, Toddler Taken Into Custody\nJacksonville, Ark., police arrested reality TV stars Joshua Rendon and Ebony Jackson-Rendon this week after police found their filthy home contained drug paraphernalia and synthetic marijuana. The state took custody of the couple\'s young child.</p><p>The 19-year-olds, who currently live at the Little Rock Air Force base, appeared on the first season of MTV\'s "16 and Pregnant" in a plot line that included Joshua Rendon enlisting in the Air Force.</p><p>Mug shots of Ebony Jackson-Rendon and Joshua Rendon provided by Jacksonville, Ark., Police Department.</p><p>When the police executed a search warrant of the Rendons\' home Tuesday, detectives described its condition as "deplorable. According to the police report, "Every room insi

In [10]:
data['instances'][0]['instance']

'### Generated Summary\n\ncollection of all usatoday.com coverage of war of chenda , including articles , videos , photos , and quotes . and videos .\n\n### Source Article\n\n\'16 & Pregnant\' Couple Arrested, Toddler Taken Into Custody\nJacksonville, Ark., police arrested reality TV stars Joshua Rendon and Ebony Jackson-Rendon this week after police found their filthy home contained drug paraphernalia and synthetic marijuana. The state took custody of the couple\'s young child.</p><p>The 19-year-olds, who currently live at the Little Rock Air Force base, appeared on the first season of MTV\'s "16 and Pregnant" in a plot line that included Joshua Rendon enlisting in the Air Force.</p><p>Mug shots of Ebony Jackson-Rendon and Joshua Rendon provided by Jacksonville, Ark., Police Department.</p><p>When the police executed a search warrant of the Rendons\' home Tuesday, detectives described its condition as "deplorable. According to the police report, "Every room inside the residence had hu

In [15]:
instances = data["instances"]


# Flatten the nested structure for easier analysis
flattened_data = []
for instance in instances:
    row = {
        "id": instance["id"],
        "instance": instance["instance"],
        #informativeness
        "informativeness_mean": instance['annotations']["Informativeness"]["mean_human"],
        "informativeness_scores": instance['annotations']["Informativeness"]["individual_human_scores"],
        #relevance
        "relevance_mean": instance['annotations']["Relevance"]["mean_human"],
        "relevance_scores": instance['annotations']["Relevance"]["individual_human_scores"],
        #fluency
        "fluency_mean": instance['annotations']["Fluency"]["mean_human"],
        "fluency_scores": instance['annotations']["Fluency"]["individual_human_scores"],
        #coherence
        "coherence_mean": instance['annotations']["Coherence"]["mean_human"],
        "coherence_scores": instance['annotations']["Coherence"]["individual_human_scores"],
    }
    flattened_data.append(row)


df = pd.DataFrame(flattened_data)
#df['n_annotations'] = df['individual_human_scores'].apply(len)

print(df.shape)
df.head()

(420, 10)


,id,instance,informativeness_mean,informativeness_scores,relevance_mean,relevance_scores,fluency_mean,fluency_scores,coherence_mean,coherence_scores
0,1,### Generated Summary\n\ncollection of all usa...,2.67,"[4, 3, 1]",3.33,"[4, 5, 1]",3.67,"[3, 5, 3]",3.67,"[4, 4, 3]"
1,2,"### Generated Summary\n\nJacksonville , Ark. ,...",4.33,"[4, 5, 4]",4.67,"[4, 5, 5]",4.33,"[3, 5, 5]",4.00,"[3, 5, 4]"
2,3,"### Generated Summary\n\nJacksonville , Ark. ,...",4.00,"[3, 5, 4]",4.00,"[4, 5, 3]",4.00,"[4, 5, 3]",4.33,"[4, 5, 4]"
3,4,### Generated Summary\n\nstars joshua rendon 1...,3.00,"[3, 3, 3]",3.67,"[3, 4, 4]",3.00,"[3, 2, 4]",2.67,"[3, 2, 3]"
4,5,### Generated Summary\n\njoshua rendon and ebo...,4.00,"[3, 4, 5]",3.33,"[4, 3, 3]",3.67,"[4, 3, 4]",3.67,"[3, 4, 4]"


In [16]:
df['instance'][0]

'### Generated Summary\n\ncollection of all usatoday.com coverage of war of chenda , including articles , videos , photos , and quotes . and videos .\n\n### Source Article\n\n\'16 & Pregnant\' Couple Arrested, Toddler Taken Into Custody\nJacksonville, Ark., police arrested reality TV stars Joshua Rendon and Ebony Jackson-Rendon this week after police found their filthy home contained drug paraphernalia and synthetic marijuana. The state took custody of the couple\'s young child.</p><p>The 19-year-olds, who currently live at the Little Rock Air Force base, appeared on the first season of MTV\'s "16 and Pregnant" in a plot line that included Joshua Rendon enlisting in the Air Force.</p><p>Mug shots of Ebony Jackson-Rendon and Joshua Rendon provided by Jacksonville, Ark., Police Department.</p><p>When the police executed a search warrant of the Rendons\' home Tuesday, detectives described its condition as "deplorable. According to the police report, "Every room inside the residence had hu